In [ ]:
%%sql -r dataframe_1
-- =====================================================
-- PHASE 3A: MEMBER STAGING TABLE
-- NOTEBOOK: NB_03_MEMBER_PROCESSING
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA STG;

CREATE TABLE IF NOT EXISTS STG_MEMBER_PROFILES (
    -- Member attributes
    MEMBER_NAME             VARCHAR(255),
    MEMBER_ID               VARCHAR(18),
    ENROLLMENT_DATE         DATE,
    LAST_FLIGHT_DATE        DATE,
    TIER_CODE               CHAR(5),
    AGENT_NAME              VARCHAR(255),
    STATE                   CHAR(5),
    COUNTRY                 CHAR(5),
    POST_CODE               NUMBER(5,0),
    DOB                     DATE,
    ACTIVE_MEMBER           CHAR(1),

    -- Derived attributes
    AGE                     NUMBER(3,0),
    STALE_MEMBER            BOOLEAN,

    -- Source metadata
    SOURCE_FILE_NAME        VARCHAR(500),
    FILE_TIMESTAMP          TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP     TIMESTAMP_NTZ,
    BATCH_ID                VARCHAR(100),
    SOURCE_ROW_NUMBER       NUMBER(38,0),

    -- Data quality fields
    VALIDATION_STATUS       VARCHAR(20),
    REJECTION_REASON        VARCHAR(2000),
    PROCESSED_TIMESTAMP     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Verify staging table
DESCRIBE TABLE STG_MEMBER_PROFILES;

SHOW TABLES IN SCHEMA STG;

In [ ]:
%%sql -r dataframe_2
-- =====================================================
-- PHASE 3B: CORE MEMBER TABLES
-- NOTEBOOK: NB_03_MEMBER_PROCESSING
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA CORE;

-- 1. Central member master table
CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE (
    MEMBER_NAME                 VARCHAR(255) NOT NULL,
    MEMBER_ID                   VARCHAR(18) NOT NULL,
    ENROLLMENT_DATE             DATE NOT NULL,
    LAST_FLIGHT_DATE            DATE,
    TIER_CODE                   CHAR(5),
    AGENT_NAME                  VARCHAR(255),
    STATE                       CHAR(5),
    COUNTRY                     CHAR(5),
    POST_CODE                   NUMBER(5,0),
    DOB                         DATE,
    ACTIVE_MEMBER               CHAR(1),

    AGE                         NUMBER(3,0),
    STALE_MEMBER                BOOLEAN,

    SOURCE_FILE_NAME            VARCHAR(500),
    FILE_TIMESTAMP              TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP         TIMESTAMP_NTZ,
    BATCH_ID                    VARCHAR(100),
    SOURCE_ROW_NUMBER           NUMBER(38,0),

    RECORD_CREATED_TIMESTAMP    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    RECORD_UPDATED_TIMESTAMP    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- 2. Country-specific member tables
CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_IND
    LIKE CORE.MEMBER_PROFILE;

CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_USA
    LIKE CORE.MEMBER_PROFILE;

CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_PHIL
    LIKE CORE.MEMBER_PROFILE;

CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_CAN
    LIKE CORE.MEMBER_PROFILE;

CREATE TABLE IF NOT EXISTS CORE.MEMBER_PROFILE_AU
    LIKE CORE.MEMBER_PROFILE;

-- 3. Verify CORE tables
SHOW TABLES IN SCHEMA CORE;

DESCRIBE TABLE CORE.MEMBER_PROFILE;

In [ ]:
-- CELL 2: RAW TO STG MEMBER TRANSFORMATION

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

INSERT INTO STG.STG_MEMBER_PROFILES (
    MEMBER_NAME, MEMBER_ID, ENROLLMENT_DATE, LAST_FLIGHT_DATE,
    TIER_CODE, AGENT_NAME, STATE, COUNTRY, POST_CODE, DOB,
    ACTIVE_MEMBER, AGE, STALE_MEMBER,
    SOURCE_FILE_NAME, FILE_TIMESTAMP, INGESTION_TIMESTAMP,
    BATCH_ID, SOURCE_ROW_NUMBER, VALIDATION_STATUS, REJECTION_REASON
)
WITH PARSED AS (
    SELECT
        RAW_RECORD:MEMBER_NAME::VARCHAR(255) AS MEMBER_NAME,
        RAW_RECORD:MEMBER_ID::VARCHAR(18) AS MEMBER_ID,
        TRY_TO_DATE(RAW_RECORD:ENROLLMENT_DATE::VARCHAR, 'YYYYMMDD') AS ENROLLMENT_DATE,
        TRY_TO_DATE(RAW_RECORD:LAST_FLIGHT_DATE::VARCHAR, 'YYYYMMDD') AS LAST_FLIGHT_DATE,
        RAW_RECORD:TIER_CODE::VARCHAR(5) AS TIER_CODE,
        RAW_RECORD:AGENT_NAME::VARCHAR(255) AS AGENT_NAME,
        RAW_RECORD:STATE::VARCHAR(5) AS STATE,
        RAW_RECORD:COUNTRY::VARCHAR(5) AS COUNTRY,
        TRY_TO_NUMBER(RAW_RECORD:POST_CODE::VARCHAR) AS POST_CODE,

        -- Assumption: DOB is supplied as DDMMYYYY
        TRY_TO_DATE(RAW_RECORD:DOB::VARCHAR, 'DDMMYYYY') AS DOB,

        RAW_RECORD:ACTIVE_MEMBER::VARCHAR(1) AS ACTIVE_MEMBER,
        SOURCE_FILE_NAME,
        FILE_TIMESTAMP,
        INGESTION_TIMESTAMP,
        BATCH_ID,
        SOURCE_ROW_NUMBER
    FROM RAW.RAW_MEMBER_PROFILES
    WHERE SOURCE_FILE_NAME = 'MEMBER_PROFILE_20261125_193100.csv'
)
SELECT
    p.MEMBER_NAME,
    p.MEMBER_ID,
    p.ENROLLMENT_DATE,
    p.LAST_FLIGHT_DATE,
    p.TIER_CODE,
    p.AGENT_NAME,
    p.STATE,
    p.COUNTRY,
    p.POST_CODE,
    p.DOB,
    p.ACTIVE_MEMBER,

    CASE
        WHEN p.DOB IS NULL THEN NULL
        ELSE DATEDIFF(YEAR, p.DOB, CURRENT_DATE())
             - IFF(
                 TO_CHAR(CURRENT_DATE(), 'MMDD') < TO_CHAR(p.DOB, 'MMDD'),
                 1, 0
             )
    END AS AGE,

    CASE
        WHEN p.LAST_FLIGHT_DATE IS NULL THEN NULL
        WHEN DATEDIFF(DAY, p.LAST_FLIGHT_DATE, CURRENT_DATE()) > 90 THEN TRUE
        ELSE FALSE
    END AS STALE_MEMBER,

    p.SOURCE_FILE_NAME,
    p.FILE_TIMESTAMP,
    p.INGESTION_TIMESTAMP,
    p.BATCH_ID,
    p.SOURCE_ROW_NUMBER,

    CASE
        WHEN p.MEMBER_NAME IS NULL OR TRIM(p.MEMBER_NAME) = '' THEN 'REJECTED'
        WHEN p.MEMBER_ID IS NULL OR TRIM(p.MEMBER_ID) = '' THEN 'REJECTED'
        WHEN p.ENROLLMENT_DATE IS NULL THEN 'REJECTED'
        ELSE 'VALID'
    END AS VALIDATION_STATUS,

    CASE
        WHEN p.MEMBER_NAME IS NULL OR TRIM(p.MEMBER_NAME) = ''
            THEN 'MEMBER_NAME_MISSING'
        WHEN p.MEMBER_ID IS NULL OR TRIM(p.MEMBER_ID) = ''
            THEN 'MEMBER_ID_MISSING'
        WHEN p.ENROLLMENT_DATE IS NULL
            THEN 'INVALID_ENROLLMENT_DATE'
        ELSE NULL
    END AS REJECTION_REASON

FROM PARSED p
WHERE NOT EXISTS (
    SELECT 1
    FROM STG.STG_MEMBER_PROFILES s
    WHERE s.SOURCE_FILE_NAME = p.SOURCE_FILE_NAME
      AND s.SOURCE_ROW_NUMBER = p.SOURCE_ROW_NUMBER
);

In [ ]:
SELECT
    MEMBER_NAME,
    MEMBER_ID,
    COUNTRY,
    DOB,
    AGE,
    STALE_MEMBER,
    VALIDATION_STATUS,
    SOURCE_FILE_NAME,
    BATCH_ID
FROM STG.STG_MEMBER_PROFILES
ORDER BY SOURCE_ROW_NUMBER;

In [ ]:
-- CELL 3: MEMBER DATA QUALITY CHECKS

USE DATABASE SKYPOINTS_INCU;

-- 1. Overall staging summary
SELECT
    COUNT(*) AS TOTAL_RECORDS,
    COUNT_IF(VALIDATION_STATUS = 'VALID') AS VALID_RECORDS,
    COUNT_IF(VALIDATION_STATUS = 'REJECTED') AS REJECTED_RECORDS,
    COUNT(DISTINCT MEMBER_ID) AS DISTINCT_MEMBER_IDS,
    COUNT(DISTINCT MEMBER_NAME) AS DISTINCT_MEMBER_NAMES
FROM STG.STG_MEMBER_PROFILES
WHERE SOURCE_FILE_NAME = 'MEMBER_PROFILE_20261125_193100.csv';

-- 2. Duplicate MEMBER_ID check
SELECT
    MEMBER_ID,
    COUNT(*) AS RECORD_COUNT
FROM STG.STG_MEMBER_PROFILES
WHERE SOURCE_FILE_NAME = 'MEMBER_PROFILE_20261125_193100.csv'
GROUP BY MEMBER_ID
HAVING COUNT(*) > 1;

-- 3. Duplicate MEMBER_NAME check (assessment key column)
SELECT
    MEMBER_NAME,
    COUNT(*) AS RECORD_COUNT
FROM STG.STG_MEMBER_PROFILES
WHERE SOURCE_FILE_NAME = 'MEMBER_PROFILE_20261125_193100.csv'
GROUP BY MEMBER_NAME
HAVING COUNT(*) > 1;

-- 4. Country-code check
SELECT
    MEMBER_ID,
    MEMBER_NAME,
    COUNTRY
FROM STG.STG_MEMBER_PROFILES
WHERE SOURCE_FILE_NAME = 'MEMBER_PROFILE_20261125_193100.csv'
  AND COUNTRY NOT IN ('IND', 'USA', 'PHIL', 'CAN', 'AU');

In [ ]:
-- CELL 4: LOAD VALID MEMBERS INTO CORE MASTER

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

MERGE INTO CORE.MEMBER_PROFILE AS T
USING (
    SELECT *
    FROM STG.STG_MEMBER_PROFILES
    WHERE VALIDATION_STATUS = 'VALID'
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY MEMBER_ID
        ORDER BY FILE_TIMESTAMP DESC, INGESTION_TIMESTAMP DESC,
                 SOURCE_FILE_NAME DESC, SOURCE_ROW_NUMBER DESC
    ) = 1
) AS S
ON T.MEMBER_ID = S.MEMBER_ID

WHEN MATCHED AND (
       S.FILE_TIMESTAMP > T.FILE_TIMESTAMP
       OR (
           S.FILE_TIMESTAMP = T.FILE_TIMESTAMP
           AND S.INGESTION_TIMESTAMP > T.INGESTION_TIMESTAMP
       )
)
THEN UPDATE SET
    T.MEMBER_NAME = S.MEMBER_NAME,
    T.ENROLLMENT_DATE = S.ENROLLMENT_DATE,
    T.LAST_FLIGHT_DATE = S.LAST_FLIGHT_DATE,
    T.TIER_CODE = S.TIER_CODE,
    T.AGENT_NAME = S.AGENT_NAME,
    T.STATE = S.STATE,
    T.COUNTRY = S.COUNTRY,
    T.POST_CODE = S.POST_CODE,
    T.DOB = S.DOB,
    T.ACTIVE_MEMBER = S.ACTIVE_MEMBER,
    T.AGE = S.AGE,
    T.STALE_MEMBER = S.STALE_MEMBER,
    T.SOURCE_FILE_NAME = S.SOURCE_FILE_NAME,
    T.FILE_TIMESTAMP = S.FILE_TIMESTAMP,
    T.INGESTION_TIMESTAMP = S.INGESTION_TIMESTAMP,
    T.BATCH_ID = S.BATCH_ID,
    T.SOURCE_ROW_NUMBER = S.SOURCE_ROW_NUMBER,
    T.RECORD_UPDATED_TIMESTAMP = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    MEMBER_NAME, MEMBER_ID, ENROLLMENT_DATE, LAST_FLIGHT_DATE,
    TIER_CODE, AGENT_NAME, STATE, COUNTRY, POST_CODE, DOB,
    ACTIVE_MEMBER, AGE, STALE_MEMBER,
    SOURCE_FILE_NAME, FILE_TIMESTAMP, INGESTION_TIMESTAMP,
    BATCH_ID, SOURCE_ROW_NUMBER
)
VALUES (
    S.MEMBER_NAME, S.MEMBER_ID, S.ENROLLMENT_DATE, S.LAST_FLIGHT_DATE,
    S.TIER_CODE, S.AGENT_NAME, S.STATE, S.COUNTRY, S.POST_CODE, S.DOB,
    S.ACTIVE_MEMBER, S.AGE, S.STALE_MEMBER,
    S.SOURCE_FILE_NAME, S.FILE_TIMESTAMP, S.INGESTION_TIMESTAMP,
    S.BATCH_ID, S.SOURCE_ROW_NUMBER
);

-- Verify loaded records
SELECT
    MEMBER_ID,
    MEMBER_NAME,
    COUNTRY,
    AGE,
    STALE_MEMBER,
    FILE_TIMESTAMP
FROM CORE.MEMBER_PROFILE
ORDER BY MEMBER_ID;

In [ ]:
-- CELL 5: COUNTRY-WISE MEMBER TABLE SYNCHRONIZATION

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

BEGIN TRANSACTION;

-- Refresh country-specific tables from the current master
DELETE FROM CORE.MEMBER_PROFILE_IND;
DELETE FROM CORE.MEMBER_PROFILE_USA;
DELETE FROM CORE.MEMBER_PROFILE_PHIL;
DELETE FROM CORE.MEMBER_PROFILE_CAN;
DELETE FROM CORE.MEMBER_PROFILE_AU;

-- India
INSERT INTO CORE.MEMBER_PROFILE_IND
SELECT * FROM CORE.MEMBER_PROFILE
WHERE COUNTRY = 'IND';

-- USA
INSERT INTO CORE.MEMBER_PROFILE_USA
SELECT * FROM CORE.MEMBER_PROFILE
WHERE COUNTRY = 'USA';

-- Philippines
INSERT INTO CORE.MEMBER_PROFILE_PHIL
SELECT * FROM CORE.MEMBER_PROFILE
WHERE COUNTRY = 'PHIL';

-- Canada
INSERT INTO CORE.MEMBER_PROFILE_CAN
SELECT * FROM CORE.MEMBER_PROFILE
WHERE COUNTRY = 'CAN';

-- Australia
INSERT INTO CORE.MEMBER_PROFILE_AU
SELECT * FROM CORE.MEMBER_PROFILE
WHERE COUNTRY = 'AU';

COMMIT;

-- Verify country-wise distribution
SELECT 'IND' AS COUNTRY, COUNT(*) AS MEMBER_COUNT
FROM CORE.MEMBER_PROFILE_IND

UNION ALL

SELECT 'USA', COUNT(*)
FROM CORE.MEMBER_PROFILE_USA

UNION ALL

SELECT 'PHIL', COUNT(*)
FROM CORE.MEMBER_PROFILE_PHIL

UNION ALL

SELECT 'CAN', COUNT(*)
FROM CORE.MEMBER_PROFILE_CAN

UNION ALL

SELECT 'AU', COUNT(*)
FROM CORE.MEMBER_PROFILE_AU

ORDER BY COUNTRY;